# SpotMicro RL: Train and Run Walking Policy

This notebook:
1. **Trains** a PPO policy for forward locomotion on the SpotMicro quadruped.
2. **Runs** the saved policy in the MuJoCo viewer so you can watch the walking spot.

Run all cells in order. Training can be shortened (e.g. `total_timesteps=50_000`) for a quick check, or increased for better walking.

## 1. Setup and training

In [5]:
import os
import gymnasium as gym
from stable_baselines3 import PPO

from spotmicro_env import SpotMicroEnv

# Training config (reduce total_timesteps for a quicker run)
TOTAL_TIMESTEPS = 100_000
SAVE_PATH = "./logs/spotmicro_ppo_notebook"
MAX_EPISODE_STEPS = 600_000  # long episodes (truncate after 600k steps)
SEED = 0

In [6]:
env = SpotMicroEnv(
    max_episode_steps=MAX_EPISODE_STEPS,
    render_mode=None,
)
env = gym.wrappers.TimeLimit(env, max_episode_steps=MAX_EPISODE_STEPS)
env = gym.wrappers.RecordEpisodeStatistics(env)

model = PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    verbose=1,
    seed=SEED,
)

os.makedirs(os.path.dirname(SAVE_PATH) or ".", exist_ok=True)
model.learn(total_timesteps=TOTAL_TIMESTEPS)
model.save(SAVE_PATH)
env.close()
print(f"Saved model to {SAVE_PATH}")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -6.33    |
| time/              |          |
|    fps             | 4733     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1e+03       |
|    ep_rew_mean          | -6.3        |
| time/                   |             |
|    fps                  | 3035        |
|    iterations           | 2           |
|    time_elapsed         | 1           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.007394709 |
|    clip_fraction        | 0.0528      |
|    clip_range           | 0.2         |
|    entropy_loss   

## 2. Run the walking spot (viewer)

Load the trained policy and run one episode with the MuJoCo viewer. Close the viewer window when done.

**On macOS:** The MuJoCo viewer requires `mjpython`. Either use a Jupyter kernel that runs under `mjpython`, or after training run from the terminal: `cd mujuco && mjpython spotmicro_walk.py` (uses the same saved model path).

In [3]:
from stable_baselines3 import PPO

# Load the model we just saved (or set LOAD_PATH to another checkpoint)
LOAD_PATH = SAVE_PATH  # or e.g. "./logs/spotmicro_ppo"
model = PPO.load(LOAD_PATH)

In [4]:
# Use render_mode="human" to show the viewer (on macOS requires mjpython kernel or run spotmicro_walk.py)
import time
env = SpotMicroEnv(
    max_episode_steps=MAX_EPISODE_STEPS,
    render_mode="human",
)
env = gym.wrappers.TimeLimit(env, max_episode_steps=MAX_EPISODE_STEPS)
# Sim time per env step; PLAYBACK_SPEED = 25 so 1s sim in 0.04s wall (no slow-mo look)
base_env = env.env if hasattr(env, "env") else env
step_sim_dt = base_env.model.opt.timestep * base_env.frame_skip
PLAYBACK_SPEED = 25.0  # 1 = real time; 25 = 25x (raise if still looks slow)

obs, info = env.reset(seed=SEED)
total_reward = 0.0
steps = 0
try:
    while True:
        t0 = time.perf_counter()
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        steps += 1
        elapsed = time.perf_counter() - t0
        time.sleep(max(0.0, step_sim_dt / PLAYBACK_SPEED - elapsed))  # playback throttle
        if terminated or truncated:
            break
except RuntimeError as e:
    if "mjpython" in str(e):
        print("Viewer requires mjpython on macOS. Run from terminal instead:")
        print("  cd mujuco && mjpython spotmicro_walk.py")
    else:
        raise
finally:
    env.close()
sim_time_s = steps * step_sim_dt
print(f"Episode finished: steps={steps}, sim_time={sim_time_s:.1f}s, total_reward={total_reward:.2f}")

RuntimeError: `launch_passive` requires that the Python script be run under `mjpython` on macOS